In [68]:
import requests
from bs4 import BeautifulSoup
import json
import re
import numpy as np

def get_product_type(node, name):
    """
    Determines if the recipe is a 'Bread', 'Cookie', 'Cake', etc.
    """
    # 1. Check metadata categories first
    categories = node.get("recipeCategory", "")
    if isinstance(categories, list):
        categories = " ".join(categories)
    
    # 2. Check keywords
    keywords = node.get("keywords", "")
    if isinstance(keywords, list):
        keywords = " ".join(keywords)
    
    search_text = f"{name} {categories} {keywords}".lower()
    
    # 3. Logic-based mapping
    mappings = {
        "Bread": ["bread", "bagel", "sourdough", "roll", "bun", "focaccia", "dough"],
        "Cookie": ["cookie", "biscuit", "macaron"],
        "Cake": ["cake", "cupcake", "torte"],
        "Pie": ["pie", "tart", "galette"],
        "Pastry": ["muffin", "scone", "croissant", "danish"]
    }
    
    for product_type, terms in mappings.items():
        if any(term in search_text for term in terms):
            return product_type
            
    return "Other"

def parse_ingredient(ing_string):
    """
    Parses '150 grams active sourdough starter' into 
    ['active sourdough starter', 150.0, 'grams']
    """
    # Pattern to match: (Quantity) (Unit) (Item)
    # Handles integers, decimals, and fractions like 1/2
    pattern = r"([0-9\s\./]+)\s*([a-zA-Z\.]+)?\s*(.*)"
    match = re.match(pattern, ing_string)
    
    if match:
        qty_raw = match.group(1).strip()
        unit = match.group(2).strip() if match.group(2) else ""
        item = match.group(3).strip()
        
        # Convert fractions or strings to float
        try:
            if '/' in qty_raw:
                num, den = qty_raw.split('/')
                qty = float(num) / float(den)
            else:
                qty = float(qty_raw)
        except ValueError:
            qty = qty_raw # Fallback to string if it's not a simple number
            
        return [item, qty, unit]
    
    return [ing_string, "", ""] # Fallback if regex fails

def scrape_recipe_v2(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        json_scripts = soup.find_all('script', type='application/ld+json')
        
        for script in json_scripts:
            try:
                data = json.loads(script.string)
                nodes = data.get('@graph', [data]) if isinstance(data, dict) else data
                
                for node in nodes:
                    if isinstance(node, dict) and 'Recipe' in str(node.get('@type')):
                        # 1. Parse Ingredients into Triplets
                        raw_ingredients = node.get("recipeIngredient", [])
                        triplets = [parse_ingredient(i) for i in raw_ingredients]
                        
                        # 2. Robust Instruction Extraction
                        raw_steps = node.get("recipeInstructions", [])
                        instructions = []

                        # 3. Get product name

                        name = node.get("name")

                        # New "Type" Key logic
                        product_type = get_product_type(node, name)
                        
                        for step in raw_steps:
                            if isinstance(step, dict):
                                # Handles 'HowToStep' or 'HowToSection'
                                if step.get('@type') == 'HowToStep':
                                    instructions.append(step.get('text', ''))
                                elif 'itemListElement' in step:
                                    for sub in step['itemListElement']:
                                        instructions.append(sub.get('text', ''))
                            else:
                                instructions.append(str(step))
                        
                        Ingredients = []
                        Amounts = []
                        Measures = []
                        for item in triplets:
                            Ingredients.append(item[0])
                            Amounts.append(item[1])
                            Measures.append(item[2])
                        Amounts = [amt if amt != '' else np.nan for amt in Amounts]
                        print(Amounts)
                        return {
                            "name": [node.get("name")],
                            "Type": [product_type],
                            "Ingredients": Ingredients,
                            "Amount" : Amounts,
                            "Measures" : Measures,
                            "Instructions": [i for i in instructions if i] # Remove empty strings
                        }
            except:
                continue
    except Exception as e:
        return f"Error: {e}"

# Test with the URL
# url = "https://simplicityandastarter.com/sourdough-bagels/"
# url = 'https://anitalianinmykitchen.com/best-pizza-dough/'

url = input("What website do you want scraped? ")
recipe = scrape_recipe_v2(url)

# Previewing the triplet format
# for ing in recipe['ingredients']:
#     print(f"Item: {ing[0]} | Qty: {ing[1]} | Unit: {ing[2]}")

# print(recipe)





[2.0, 1.0, 1.0, 2.0, 5.0, 1.0]


In [74]:
import pandas as pd
# 1. Calculate how many blanks are needed
target_length = len(recipe['Ingredients'])
if len(recipe['Instructions']) > target_length:
    target_length = len(recipe['Instructions'])

ins_padding = target_length - len(recipe['Instructions'])
name_padding = target_length - len(recipe['name'])
type_padding = target_length - len(recipe['Type'])
ing_padding = target_length - len(recipe['Ingredients'])
amount_padding = target_length - len(recipe['Amount'])
measure_padding = target_length - len(recipe['Measures'])

# 2. Append the blank strings
# This creates a list like ['', '', '', ...] and adds it to the end
ins_padded_instructions = recipe['Instructions'] + ([''] * ins_padding)
name_padded_instructions = recipe['name'] + ([''] * name_padding)
type_padded_instructions = recipe['Type'] + ([''] * type_padding)
ing_padded_instructions = recipe['Ingredients'] + ([''] * ing_padding)
amount_padded_instructions = recipe['Amount'] + ([''] * amount_padding)
measure_padded_instructions = recipe['Measures'] + ([''] * measure_padding)

df = pd.DataFrame({
    'Recipe' : name_padded_instructions,
    'Type' : type_padded_instructions,
    'Ingredients': ing_padded_instructions,
    'Amount': amount_padded_instructions,
    'Measure': measure_padded_instructions,
    'Instructions' : ins_padded_instructions
})

# 2. Add the single-item info to every row

# Now it builds perfectly!
print(df)

# Save it
df.to_csv(r'C:\Users\Brad Hepburn\Desktop\test.csv', index=False)

                       Recipe   Type                              Ingredients  \
0   Best Homemade Pizza Dough  Bread                           lukewarm water   
1                                                                       sugar   
2                                              ½ tablespoons active dry yeast   
3                                                                   olive oil   
4                                     ¼ cups all purpose flour or bread flour   
5                                                            ½ teaspoons salt   
6                                                                               
7                                                                               
8                                                                               
9                                                                               
10                                                                              

   Amount      Measure     